In [ ]:
import json
import pandas as pd

# 1. Single Object JSON {}
single_json = {
    "emp_id": 101,
    "name": "Alex",
    "department": "Data Engineering",
    "location": {"city": "Pune", "country": "India"},
    "skills": ["Python", "SQL", "Snowflake"]
}

# 2. Array of Objects JSON [{}]
array_json = [
    {
        "emp_id": 101,
        "name": "Alex",
        "department": "Data Engineering",
        "location": {"city": "Pune", "country": "India"},
        "skills": ["Python", "SQL"]
    },
    {
        "emp_id": 102,
        "name": "Sam",
        "department": "Analytics",
        "location": {"city": "Ahmedabad", "country": "India"},
        "skills": ["Tableau", "dbt"]
    }
]

# Write files to disk
with open("single_object.json", "w") as f:
    json.dump(single_json, f, indent=4)

with open("array_objects.json", "w") as f:
    json.dump(array_json, f, indent=4)

LOAD / READ OPERATIONS - Single {} json object

In [ ]:
# Method 1: Direct Pandas Read (using series conversion)
# as json is having mix of dictionaries and other non-series objects (like lists) we need to frame it and transform
df_single_m1 = pd.read_json("single_object.json", typ="series").to_frame().T

print(df_single_m1.to_string())

  emp_id  name        department                              location                    skills
0    101  Alex  Data Engineering  {'city': 'Pune', 'country': 'India'}  [Python, SQL, Snowflake]


In [25]:
# Method 2: Standard json.load() + DataFrame
with open("single_object.json") as f:
    data_single = json.load(f)
df_single_m2 = pd.DataFrame([data_single])
print(df_single_m2.to_string())


   emp_id  name        department                              location                    skills
0     101  Alex  Data Engineering  {'city': 'Pune', 'country': 'India'}  [Python, SQL, Snowflake]


LOAD / READ OPERATIONS - Array [{}] json object

In [26]:
# Method 1: Direct Pandas Read
df_array_m1 = pd.read_json("array_objects.json")
print(df_array_m1.to_string())

   emp_id  name        department                                   location          skills
0     101  Alex  Data Engineering       {'city': 'Pune', 'country': 'India'}   [Python, SQL]
1     102   Sam         Analytics  {'city': 'Ahmedabad', 'country': 'India'}  [Tableau, dbt]


In [27]:
# Method 2: Standard json.load() + DataFrame
with open("array_objects.json") as f:
    data_array = json.load(f)
df_array_m2 = pd.DataFrame(data_array)
print(df_array_m2.to_string())

   emp_id  name        department                                   location          skills
0     101  Alex  Data Engineering       {'city': 'Pune', 'country': 'India'}   [Python, SQL]
1     102   Sam         Analytics  {'city': 'Ahmedabad', 'country': 'India'}  [Tableau, dbt]


Load Dynamically and Normalize dynamically as per file data

In [39]:
def load_json_dynamically(file_path):
    with open(file_path, "r") as f:
        data = json.load(f)

    # Check top-level type
    if isinstance(data, dict):
        print(f"[{file_path}] Detected: Single JSON Object {{}}")
        df = pd.DataFrame([data_single])
        df_norm = pd.json_normalize(data)
        '''
        loc_df_single = pd.json_normalize(df_single_m1["location"])
        df.drop(columns=["location"]).join(loc_df_single)
        '''
    elif isinstance(data, list):
        print(f"[{file_path}] Detected: Array of JSON Objects [{{}}]")
        df = pd.DataFrame([data_single])
        df_norm = pd.json_normalize(data)
        '''
        loc_df_array = df_array_m1["location"].apply(pd.Series)
        df_flat_array_m2 = pd.concat([df_array_m1.drop(columns=["location"]), loc_df_array], axis=1)
        '''
    else:
        raise ValueError("Unsupported JSON structure")

    return df, df_norm


df, df_norm = load_json_dynamically("single_object.json")
print('----------------Original--------------------------------------------')
print(df.to_string())
print('----------------Normalized------------------------------------------')
print(df_norm.to_string())
print('-----------------------------------------------------------------------------------------')
print('----------------Original--------------------------------------------')
df_array = load_json_dynamically("array_objects.json")
print(df.to_string())
print('----------------Normalized------------------------------------------')
print(df_norm.to_string())
print('-----------------------------------------------------------------------------------------')

[single_object.json] Detected: Single JSON Object {}
----------------Original--------------------------------------------
   emp_id  name        department                              location                    skills
0     101  Alex  Data Engineering  {'city': 'Pune', 'country': 'India'}  [Python, SQL, Snowflake]
----------------Normalized------------------------------------------
   emp_id  name        department                    skills location.city location.country
0     101  Alex  Data Engineering  [Python, SQL, Snowflake]          Pune            India
-----------------------------------------------------------------------------------------
----------------Original--------------------------------------------
[array_objects.json] Detected: Array of JSON Objects [{}]
   emp_id  name        department                              location                    skills
0     101  Alex  Data Engineering  {'city': 'Pune', 'country': 'India'}  [Python, SQL, Snowflake]
----------------No